In [3]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [111]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        ##cs.execute('SELECT COUNT(*) AS invalid_dates FROM orders o WHERE o."date" <= 0')
        ##print(cs.fetchall())
        runQueries(cs)
        ##cs.execute('SELECT * FROM "v_cust_purch_summary_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[('View "v_cust_purch_summary_f" successfully created.',)]


In [103]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "v_cust_purch_summary_f" AS 
    SELECT 
    c."customer_id", 
    c."customer_name", 
    COUNT(
        CASE 
            WHEN o."order_id" IS NOT NULL THEN o."order_id"
            ELSE 0 
        END
    ) AS total_orders,
    SUM(
        CASE 
            WHEN o."total_price" IS NOT NULL THEN o."total_price"
            ELSE 0 
        END) AS total_money_spent, 
    MIN(
        CASE 
            WHEN o."date" > 0 THEN TO_DATE(TO_TIMESTAMP(o."date")) 
            ELSE NULL 
        END
    ) AS first_order_date,
    AVG(
        CASE 
            WHEN o."total_price" IS NOT NULL THEN o."total_price"
            ELSE NULL 
        END) AS avg_money_spent,
    AVG(CASE 
            WHEN o."profit" IS NOT NULL THEN o."profit"
            ELSE NULL 
        END) AS avg_money_earned_per_order 
    FROM customers c 
    JOIN orders o 
    ON c."customer_id" = o."customer_id"
    WHERE o."date" > 0
    GROUP BY c."customer_id", c."customer_name";
    """
    cs.execute(query)